# 05 · Batch Forecasting — Many Series at Once

TimesFM forecasts **hundreds or thousands of series in one call**. This is the
key to building a scalable service. Just pass a list of arrays — they can have
**different lengths**.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# Simulate 8 stores, each with a different history length & pattern
rng = np.random.default_rng(7)
inputs = []
names = [f"store_{i:02d}" for i in range(8)]
for i in range(8):
    n = rng.integers(120, 260)                      # varying length
    t = np.arange(n)
    level = rng.uniform(50, 500)
    s = (level
         + level * 0.15 * np.sin(2*np.pi*t/52 + rng.uniform(0, 6))
         + rng.normal(0, level*0.04, size=n)).astype(np.float32)
    inputs.append(np.clip(s, 0, None))

for nm, s in zip(names, inputs):
    print(f"{nm}: {s.size} points")

In [ ]:
horizon = 30
point, q = model.forecast(horizon=horizon, inputs=inputs)
print("point:", point.shape)   # (8, 30)
print("quant:", q.shape)       # (8, 30, 10)

## Export all forecasts to JSON (ready for an API response)

In [ ]:
import json

IDX_Q10, IDX_Q50, IDX_Q90 = 1, 5, 9
results = {
    name: {
        "median":   point[i].round(2).tolist(),
        "lower_80": q[i, :, IDX_Q10].round(2).tolist(),
        "upper_80": q[i, :, IDX_Q90].round(2).tolist(),
    }
    for i, name in enumerate(names)
}
with open("batch_forecasts.json", "w") as f:
    json.dump(results, f, indent=2)
print("wrote batch_forecasts.json for", len(results), "series")
print(json.dumps({names[0]: results[names[0]]}, indent=2)[:400], "...")

## Grid of small multiples

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=False)
for i, ax in enumerate(axes.ravel()):
    hist = inputs[i][-60:]
    xf = range(len(hist), len(hist) + horizon)
    ax.plot(range(len(hist)), hist, color="tab:blue")
    ax.plot(xf, point[i], color="tab:orange")
    ax.fill_between(xf, q[i, :, IDX_Q10], q[i, :, IDX_Q90], alpha=0.2, color="tab:orange")
    ax.set_title(names[i], fontsize=9)
fig.suptitle("Batch forecast — 8 stores")
fig.tight_layout(); fig.savefig("batch_grid.png", dpi=120)
print("saved batch_grid.png")

### Performance tip
For very large batches, set `per_core_batch_size` in the config and/or process
in chunks:

```python
CHUNK = 50
for i in range(0, len(inputs), CHUNK):
    p, qq = model.forecast(horizon=horizon, inputs=inputs[i:i+CHUNK])
```